# ETF Pair Deep Dive

Change `TICKER_A` and `TICKER_B` below to inspect any pair that exists in the current data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

TICKER_A = "SCHQ"
TICKER_B = "SPTL"

cwd = Path.cwd()
if cwd.name == "notebooks":
    project_dir = cwd.parent
elif (cwd / "pairs_trading").exists():
    project_dir = cwd / "pairs_trading"
else:
    project_dir = cwd

data_dir = project_dir / "data"
outputs_dir = project_dir / "outputs"
pair_name = f"{TICKER_A}_{TICKER_B}"

price_history_path = data_dir / "etf_price_history.csv"
scan_path = outputs_dir / "cointegration_scan.csv"
backtest_summary_path = outputs_dir / f"backtest_{pair_name}_summary.csv"
backtest_trades_path = outputs_dir / f"backtest_{pair_name}_trades.csv"
backtest_daily_path = outputs_dir / f"backtest_{pair_name}_daily.csv"

price_history_path, scan_path, backtest_summary_path, backtest_trades_path, backtest_daily_path

## Price History

In [ ]:
prices = pd.read_csv(price_history_path, parse_dates=["date"])
pair_prices = (
    prices[prices["ticker"].isin([TICKER_A, TICKER_B])]
    .pivot(index="date", columns="ticker", values="adj_close")
    .dropna()
    .sort_index()
)

if pair_prices.empty or set([TICKER_A, TICKER_B]).difference(pair_prices.columns):
    raise ValueError(f"Missing price history for {TICKER_A} / {TICKER_B}")

pair_prices.tail()

In [ ]:
normalized = pair_prices / pair_prices.iloc[0] * 100
ax = normalized.plot(figsize=(11, 5), linewidth=1.4)
ax.set_title(f"{TICKER_A} / {TICKER_B} normalized adjusted close")
ax.set_ylabel("Indexed value, first date = 100")
ax.grid(True, alpha=0.25)
plt.show()

## Cointegration Scan Result

In [ ]:
scan = pd.read_csv(scan_path)
pair_scan = scan[
    ((scan["ticker_a"] == TICKER_A) & (scan["ticker_b"] == TICKER_B))
    | ((scan["ticker_a"] == TICKER_B) & (scan["ticker_b"] == TICKER_A))
]

if pair_scan.empty:
    print(f"No cointegration scan row found for {TICKER_A} / {TICKER_B}.")
else:
    display(pair_scan.T)

## Spread And Z-Score

In [ ]:
if pair_scan.empty:
    hedge_ratio = 1.0
    intercept = 0.0
    print("Using fallback hedge_ratio=1.0 and intercept=0.0 because no scan row was found.")
else:
    scan_row = pair_scan.iloc[0]
    hedge_ratio = scan_row["hedge_ratio"]
    intercept = scan_row["intercept"]

log_prices = np.log(pair_prices[[TICKER_A, TICKER_B]])
spread = log_prices[TICKER_A] - intercept - hedge_ratio * log_prices[TICKER_B]
zscore = (spread - spread.mean()) / spread.std()

pd.Series(
    {
        "hedge_ratio": hedge_ratio,
        "intercept": intercept,
        "spread_mean": spread.mean(),
        "spread_std": spread.std(),
        "spread_std_bps": spread.std() * 10_000,
        "latest_zscore": zscore.iloc[-1],
    }
)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

spread.plot(ax=axes[0], linewidth=1.2)
axes[0].axhline(spread.mean(), color="black", linestyle="--", linewidth=1)
axes[0].axhline(spread.mean() + spread.std(), color="gray", linestyle=":", linewidth=1)
axes[0].axhline(spread.mean() - spread.std(), color="gray", linestyle=":", linewidth=1)
axes[0].set_title(f"{TICKER_A} / {TICKER_B} spread")
axes[0].set_ylabel("Log spread")

zscore.plot(ax=axes[1], linewidth=1.2)
for level, color, linestyle in [(2, "crimson", "--"), (-2, "crimson", "--"), (0.5, "darkgreen", ":"), (-0.5, "darkgreen", ":"), (0, "black", "-")]:
    axes[1].axhline(level, color=color, linestyle=linestyle, linewidth=1)
axes[1].set_title(f"{TICKER_A} / {TICKER_B} z-score")
axes[1].set_ylabel("Z-score")

for ax in axes:
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

## Backtest Outputs

In [ ]:
if backtest_summary_path.exists():
    backtest_summary = pd.read_csv(backtest_summary_path)
    display(backtest_summary.T)
else:
    print(f"No backtest summary found at {backtest_summary_path}")

In [ ]:
if backtest_trades_path.exists():
    trades = pd.read_csv(backtest_trades_path, parse_dates=["entry_date", "exit_date"])
    display(trades.tail(20))
else:
    print(f"No trade list found at {backtest_trades_path}")

In [ ]:
if backtest_daily_path.exists():
    daily = pd.read_csv(backtest_daily_path, parse_dates=["date"])
    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
    axes[0].plot(daily["date"], daily["equity"], linewidth=1.2)
    axes[0].set_title(f"{TICKER_A} / {TICKER_B} backtest equity")
    axes[0].set_ylabel("Equity")
    axes[1].plot(daily["date"], daily["position"], linewidth=1.0)
    axes[1].set_title("Position")
    axes[1].set_ylabel("-1 / 0 / +1")
    for ax in axes:
        ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print(f"No daily backtest file found at {backtest_daily_path}")